## ---- Create a distort function using spacy ----- 

### 1. Import librairies

In [1]:
import spacy 
import pandas as pd
import re
import unicodedata
from collections import defaultdict

### 2. Import spacy nlp

In [2]:
nlp = spacy.load("en_core_web_trf")

C:\Users\Léna\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 3. Distort Function 

In [3]:
# Cache index so we don't reload the file each call for morphhological data
_SEG_INDEX = None

In [4]:
def distort( paragraph,
    lengths = False, # If we want "xxxx"
    punctuation =False, # keep the punctuation 
    closed_class=False, # keep closed class words 
    morphology=False, # like "blanking"
    capitals= False, # like "Blank"
    pos =False, # like "NOUN" instead of "blank"
    numbers= False,   # "num" instead of "blank"      
    reveal_numbers=False,  # show exact number
    blank_token= "blank",
    segmentation_path="eng.segmentations",
    enclitic=False # For "you'll", "don't", etc
    ):
    
    # normalize unicode (Wikipedia often has thin spaces, etc.)
    paragraph = unicodedata.normalize("NFKC", paragraph)

    doc = nlp(paragraph)
    output = []

    # No officiel closed class list for spacy so we create one
    CLOSED_CLASS_POS = {"DET", "PRON", "ADP", "AUX", "CCONJ", "SCONJ", "PART"}

    # For morphology, the file has different annotations
    POS2MORPH = {
        "NOUN": "N",
        "PROPN": "N",
        "VERB": "V",
        "AUX": "V",
    }

    seg_index = defaultdict(list)
    if morphology:
        with open(segmentation_path, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                parts = line.rstrip("\n").split("\t")
                if len(parts) < 4:
                    continue
                surface = parts[1].lower()
                morph_tag = parts[2]
                seg = parts[3]
                tag0 = morph_tag.split("|")[0].split(";")[0]
                seg_index[surface].append({"tag0": tag0, "seg": seg})

    def is_number_token(token):
        return bool(re.search(r"\d", token.text))

    def mask_x(token, char="x"):
        return char * len(token.text)

    def mask_number_pattern(token_text):
        chars = []
        for ch in token_text:
            if ch.isdigit():
                chars.append("n")
            elif ch.isalpha():
                chars.append("x")
            else:
                chars.append(ch)
        return "".join(chars)

    def apply_capitals(original, jab):
        if original and original[0].isupper() and jab:
            return jab[0].upper() + jab[1:]
        return jab

    def apply_capitals_pattern(original, jab):
        chars = list(jab)
        n = min(len(original), len(chars))
        for i in range(n):
            if original[i].isupper():
                chars[i] = chars[i].upper()
            else: 
                chars[i] = chars[i].lower()
        return "".join(chars)

    def split_enclitic(token_text):
        m = re.match(r"^(.+?)(n't|'ll|'re|'ve|'d|'m|'s|’ll|’re|’ve|’d|’m|’s)$",
                     token_text, flags=re.IGNORECASE)
        if m:
            return m.group(1), m.group(2)
        return None, None

    def is_clitic_token_text(txt):
        return txt.lower() in {
            "'ll", "'re", "'ve", "'d", "'m", "'s",
            "’ll", "’re", "’ve", "’d", "’m", "’s",
            "n't"
        }

    def morph_jabber_from_text(surface_text, token_pos):
        form = surface_text.lower()
        analyses = seg_index.get(form, [])

        target_pos = POS2MORPH.get(token_pos)
        if target_pos:
            pos_filtered = [a for a in analyses if a["tag0"] == target_pos]
            if pos_filtered:
                analyses = pos_filtered

        if not analyses:
            if lengths:
                return "x" * len(surface_text)
            return blank_token

        seg = analyses[0]["seg"]

        if seg == "-" or not seg:
            if lengths:
                return "x" * len(surface_text)
            return blank_token

        pieces = seg.split("|")

        if len(pieces) <= 1:
            if lengths:
                return "x" * len(surface_text)
            return blank_token

        stem = "".join(pieces[:-1])
        suffix = pieces[-1]

        if lengths:
            return ("x" * len(stem)) + suffix

        return blank_token + suffix

    def morph_jabber(token):
        return morph_jabber_from_text(token.text, token.pos_)

    if pos and lengths:
        raise ValueError("Incompatible: pos=True and lengths=True.")
    if pos and morphology:
        raise ValueError("Incompatible: pos=True and morphology=True.")

    i = 0
    while i < len(doc):
        token = doc[i]

        if token.is_space:
            output.append(token.whitespace_)
            i += 1
            continue

        # =========================
        # ENCLITIC BLOCK
        # =========================
        combined = False
        if enclitic and (i + 1) < len(doc):
            nxt = doc[i + 1]
            if (not token.is_space) and (not token.is_punct) and is_clitic_token_text(nxt.text):

                base_text = token.text
                clitic_text = nxt.text
                full_text = base_text + clitic_text

                if is_number_token(token):

                    if reveal_numbers:
                        jab = full_text
                    elif lengths:
                        jab = mask_number_pattern(full_text)
                        if not punctuation:
                            jab = re.sub(r"[^A-Za-z0-9]", "", jab)
                    elif numbers:
                        jab = "num" + clitic_text
                    else:
                        jab = blank_token + clitic_text

                    if pos and not (reveal_numbers and is_number_token(token)):
                        jab = token.pos_.lower()

                    if capitals:
                        if lengths:
                            jab = apply_capitals_pattern(full_text, jab)
                        else:
                            jab = apply_capitals(full_text, jab)

                elif pos:
                    jab = token.pos_.lower()
                    if capitals:
                        jab = apply_capitals(full_text, jab)

                elif closed_class and token.pos_ in CLOSED_CLASS_POS:
                    jab = full_text

                elif morphology:
                    if token.pos_ in CLOSED_CLASS_POS:
                        if lengths:
                            base_jab = "x" * len(base_text)
                        else:
                            base_jab = blank_token
                    else:
                        base_jab = morph_jabber_from_text(base_text, token.pos_)
                    jab = base_jab + clitic_text

                    if capitals:
                        jab = apply_capitals(full_text, jab)

                elif lengths:
                    jab = ("x" * len(base_text)) + clitic_text
                    if capitals:
                        jab = apply_capitals_pattern(full_text, jab)

                else:
                    jab = blank_token + clitic_text
                    if capitals:
                        jab = apply_capitals(full_text, jab)

                output.append(jab)
                output.append(nxt.whitespace_)
                i += 2
                combined = True

        if combined:
            continue

        # =========================
        # PONCTUATION BLOCK
        # =========================
        if token.text == "-":
            output.append(token.text)
            output.append(token.whitespace_)
            i += 1
            continue

        if token.is_punct or token.text in {"'", "’"}:
            if punctuation:
                output.append(token.text)
                output.append(token.whitespace_)
            else:
                if token.whitespace_:
                    output.append(token.whitespace_)
                elif output and not output[-1].endswith(" "):
                    output.append(" ")
            i += 1
            continue

        # =========================
        # NUMBERS
        # =========================
        if is_number_token(token):

            if reveal_numbers:
                jab = token.text
            elif lengths:
                jab = mask_number_pattern(token.text)
                if not punctuation:
                    jab = re.sub(r"[^A-Za-z0-9]", "", jab)
            elif numbers:
                jab = "num"
            else:
                jab = blank_token

            if pos and not (reveal_numbers and is_number_token(token)):
                jab = token.pos_.lower()

            if capitals:
                if lengths:
                    jab = apply_capitals_pattern(token.text, jab)
                else:
                    jab = apply_capitals(token.text, jab)

            output.append(jab)
            output.append(token.whitespace_)
            i += 1
            continue

        # =========================
        # POS
        # =========================
        if pos:
            jab = token.pos_.lower()
            if capitals:
                jab = apply_capitals(token.text, jab)
            output.append(jab)
            output.append(token.whitespace_)
            i += 1
            continue

        # =========================
        # CLOSED CLASS
        # =========================
        if closed_class and token.pos_ in CLOSED_CLASS_POS:
            output.append(token.text)
            output.append(token.whitespace_)
            i += 1
            continue

        # =========================
        # MORPHOLOGY
        # =========================
        if morphology:
            if token.pos_ in CLOSED_CLASS_POS:
                if lengths:
                    jab = mask_x(token)
                else:
                    jab = blank_token
            else:
                jab = morph_jabber(token)

            if capitals:
                jab = apply_capitals(token.text, jab)

            output.append(jab)
            output.append(token.whitespace_)
            i += 1
            continue

        # =========================
        # LENGTHS
        # =========================
        if lengths:
            jab = mask_x(token)
            if capitals:
                jab = apply_capitals_pattern(token.text, jab)
            output.append(jab)
            output.append(token.whitespace_)
            i += 1
            continue

        # =========================
        # BASE
        # =========================
        jab = blank_token
        if capitals:
            jab = apply_capitals(token.text, jab)

        output.append(jab)
        output.append(token.whitespace_)
        i += 1

    result = "".join(output)
    result = re.sub(r"[ \t]+", " ", result).strip()
    return result

In [17]:
distort("Before the emergence of transformer-based models in 2017, some language models were considered large relative to the computational and data constraints of their time. In the early 1990s, IBM's statistical models pioneered word alignment techniques for machine translation, laying the groundwork for corpus-based language modeling.", lengths = True, reveal_numbers= True, morphology = True, punctuation = True, enclitic=True,  capitals=True, closed_class=True)

"Before the xxxxxxxxx of xxxxxxxxxxx-xxxxed xxxxxs in 2017, some xxxxxxxx xxxxxs were xxxxxxxxxx xxxxx xxxxxxxx to the xxxxxxxxxxxxx and xxxx xxxxxxxxxxx of their xxxx. In the xxxxx 1990s, Xxx's xxxxxxxxxxx xxxxxs xxxxxxxed xxxx xxxxxxxxx xxxxxxxxxx for xxxxxxx xxxxxxxxxxx, xxxing the xxxxxxxxxx for xxxxxx-xxxxed xxxxxxxx xxxxxxxx."

In [5]:
distort("It may be flavored with dried fruit such as sultanas or raisins, or other fruits such as apples. In France and Spain, croissants are generally sold without filling and eaten without added butter, but sometimes with almond filling.", lengths = True, reveal_numbers= True, morphology = True, punctuation = True, enclitic=True,  capitals=True, closed_class=True)

'It may be xxxxxxxx with xxxed xxxxx xxxx as xxxxxxxx or xxxxxxs, or xxxxx xxxxxx xxxx as xxxxxs. In Xxxxxx and Xxxxx, xxxxxxxxxs are xxxxxxxxx xxxx without xxxxxxx and xxxen without xxxxed xxxxxx, but xxxxxxxxx with xxxxxx xxxxxxx.'

In [18]:
## Some wikipedia paragraphs to test on 

Paragraph1 = "Pythagoras of Samos[a] (Ancient Greek: Πυθαγόρας; c. 570 – c. 495 BC)[b] was an ancient Ionian Greek philosopher, polymath, and the eponymous founder of Pythagoreanism."
Paragraph2 = "Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making."
Paragraph3 = "Sir Isaac Newton (/ˈnjuːtən/ ⓘ; 4 January [O.S. 25 December] 1643 – 31 March [O.S. 20 March] 1727)[a] was an English polymath who was a mathematician, physicist, astronomer, alchemist, theologian, author and inventor.[5] He was a key figure in the Scientific Revolution and the Enlightenment that followed.[6] His book Philosophiæ Naturalis Principia Mathematica (Mathematical Principles of Natural Philosophy), first published in 1687, achieved the first great unification in physics and established classical mechanics.[7][8] Newton also made seminal contributions to optics, and shares credit with the German mathematician Gottfried Wilhelm Leibniz for formulating infinitesimal calculus, although he developed calculus years before Leibniz. Newton contributed to and refined the scientific method, and his work is considered the most influential in bringing forth modern science."


In [19]:
distort(Paragraph3, morphology = True, punctuation = True, capitals = True)


'Blank Blank Blank (blank blank; blank Blank [Blank blank Blank] blank – blank Blank [Blank blank Blank] blank] blank blank Blank blank blank blank blank blank, blank, blank, blank, blank, blank blank blank] Blank blank blank blank blank blank blank Blank Blank blank blank Blank blank blank] Blank blank Blank Blank Blank Blank (Blank Blanks blank Blank Blank), blank blanked blank blank, blanked blank blank blank blank blank blank blank blanked blank blank] Blank blank blank blank blank blank blanks, blank blanks blank blank blank Blank blank Blank Blank Blank blank blanking blank blank, blank blank blanked blank blanks blank Blank. Blank blanked blank blank blanked blank blank blank, blank blank blank blank blank blank blank blank blank blanking blank blank blank.'

### 4. Distort a csv

We take a csv that has already been cleaned (no /n and one paragraph chosen randomly), for now we have 100 articles

In [30]:
df = pd.read_csv("data/wiki_original_text.csv")

In [31]:
df.head(10)

,id,url,title,text
0,34335512,https://en.wikipedia.org/wiki/Yuknoom%20Ch%CA%BCeen%20II,Yuknoom Chʼeen II,"Yuknoom Chʼeen II (September 11, 600 – 680s), known as Yuknoom the Great, was a Maya ruler of the Kaan kingdom, which had its capital at Calakmul during the Classic Period of Mesoamerican chronology."
1,43869726,https://en.wikipedia.org/wiki/Rats%20and%20Cats,Rats and Cats,Rats and Cats is a 2007 Australian comedy film directed by Tony Rogers and starring Adam Zwar and Jason Gann.
2,52082793,https://en.wikipedia.org/wiki/Corinne%20Lagache,Corinne Lagache,Corinne Lagache (born 9 December 1975 in Caen) is a French footballer who played as a goalkeeper for the France women's national football team. She was part of the team at the UEFA Women's Euro 2001. On club level she played for La Roche ESOF in France.
3,8273434,https://en.wikipedia.org/wiki/North%20Macedonia%20men%27s%20national%20handball%20team,North Macedonia men's national handball team,"World Cup For the World cup 1995 only teams from EURO 1994 qualified so again team Macedonian did not get a chance to participate. At the World Cup's they entered 8 times – 1999, 2009, 2013, 2015, 2017, 2019, 2021 and 2023. The most successful was in 2015 when they finished 9th."
4,37897173,https://en.wikipedia.org/wiki/Scopula%20inactuosa,Scopula inactuosa,Scopula inactuosa is a moth of the family Geometridae. It is found on Sumbawa.
5,28985107,https://en.wikipedia.org/wiki/Ashleys%20Copse,Ashleys Copse,"The site is to the northeast of the village of Middle Winterslow, and falls mostly within the county of Hampshire, and partly within the county of Wiltshire."
6,29512629,https://en.wikipedia.org/wiki/Gruvleflesa%20Knolls,Gruvleflesa Knolls,"The Gruvleflesa Knolls () are two low rock knolls rising above the Antarctic glacial moraine west of the Gruvletindane Crags, in the Kurze Mountains of Queen Maud Land. They were mapped from surveys and air photos by the Sixth Norwegian Antarctic Expedition (1956–60) and named Gruvleflesa."
7,65425941,https://en.wikipedia.org/wiki/August%20Merges,August Merges,"In 1906 he was finally able to abandon tailoring work, becoming instead a salaried party official, working in nearby Hildesheim and Alfeld. Two years later, in 1908, he was elected to membership of the Delligsen municipal council. He was by now gaining a reputation in the region as an effective political orator (or ""agitator""). He took a leading role in Labour movement demonstrations, campaigning powerfully for democratic state elections, which meant campaigning against the infamous ""Dreiklassenwahlrecht"" (literally ""Three-class voting entitlement""), whereby the 80% of voters paying the lowest amounts in tax had the same levels of influence in election results as the richest 5% paying the highest amounts. (The precise proportions varied slightly over time according to tax rates and levels of prosperity, but the underlying inequality would endure till 1918.)"
8,40673733,https://en.wikipedia.org/wiki/Patricia%20Cladis,Patricia Cladis,"Patricia Elisabeth Cladis (July 13, 1937 – July 3, 2017) was a Canadian-American physicist who specialized in the physics of liquid crystals. She was a research physicist at Bell Labs in Murray Hill, New Jersey from 1972 to 1997 before founding Advanced Liquid Crystal Technologies in Summit, New Jersey. She was a fellow of the American Physical Society and also received a Guggenheim fellowship."
9,357414,https://en.wikipedia.org/wiki/North%20American%20Bus%20Industries,North American Bus Industries,"2008 also saw additional expansion at NABI's Anniston plant with the installation of a new, robotic paint system. This expansion took the Anniston plant to approximately 1/3 million square feet under roof, not including adjacent leased facilities used for manufacture of standard-floor unfinished buses."


#### 4.1. First for length 

In [32]:
df_length = df.copy()

# Apply distortion with all options enabled except POS and numbers (to avoid conflicts)
df_length["text"] = df_length["text"].apply(
    lambda x: distort(
        x,
        lengths=True,
        punctuation=True,
        closed_class=True,
        morphology=True,
        capitals=True,
        reveal_numbers=True,
        enclitic=True
    )
)

df_length = df_length[["id", "text"]]
df_length.to_csv("wiki_distorted_LPCMKRE.csv", index=False)
print("Saved distorted CSV")

Saved distorted CSV


In [33]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

In [34]:
# Just to see what it looks like
df_length = pd.read_csv("data/distorted/wiki_distorted_LPCMKRE.csv")
df_length.head(10)

,id,text
0,34335512,"Xxxxxxx Xxxxxx XX (Xxxxxxxxx 11, 600 – 680s), xxxxx as Xxxxxxx the Xxxxx, was a Xxxx xxxxx of the Xxxx xxxxxxx, which xxx its xxxxxxx at Xxxxxxxx during the Xxxxxxx Xxxxxx of Xxxxxxxxxxxx xxxxxxxxxx."
1,43869726,Xxxs and Xxxs is a 2007 Xxxxxxxxxx xxxxxx xxxx xxxxxxxx by Xxxx Xxxxxx and xxxxing Xxxx Xxxx and Xxxxx Xxxx.
2,52082793,Xxxxxxx Xxxxxxx (xxxx 9 Xxxxxxxx 1975 in Xxxx) is a Xxxxxx xxxxxxxxxx who xxxxxed as a xxxxxxxxxx for the Xxxxxx xxxxx's xxxxxxxx xxxxxxxx xxxx. She was xxxx of the xxxx at the XXXX Xxxxx's Xxxx 2001. On xxxx xxxxx she xxxxxed for Xx Xxxxx XXXX in Xxxxxx.
3,8273434,"Xxxxx Xxx For the Xxxxx xxx 1995 xxxx xxxxs from XXXX 1994 xxxxxxxed so xxxxx xxxx Xxxxxxxxxx did not xxx a xxxxxx to xxxxxxxxxxx. At the Xxxxx Xxx's they xxxxxed 8 xxxxs – 1999, 2009, 2013, 2015, 2017, 2019, 2021 and 2023. The xxxx xxxxxxxxxx was in 2015 when they xxxxxxed 9th."
4,37897173,Xxxxxxx xxxxxxxxx is a xxxx of the xxxxxx Xxxxxxxxxxx. It is xxxxx on Xxxxxxx.
5,28985107,"The xxxx is to the xxxxxxxxx of the xxxxxxx of Xxxxxx Xxxxxxxxxx, and xxxxs xxxxxx within the xxxxxx of Xxxxxxxxx, and xxxxxx within the xxxxxx of Xxxxxxxxx."
6,29512629,"The Xxxxxxxxxxx Xxxxxs () are xxx xxx xxxx xxxxxs xxxxing above the Xxxxxxxxx xxxxxxx xxxxxxx xxxx of the Xxxxxxxxxxxxx Xxxxs, in the Xxxxx Xxxxxxxxx of Xxxxx Xxxx Xxxx. They were xxxed from xxxxxxs and xxx xxxxxs by the Xxxxx Xxxxxxxxx Xxxxxxxxx Xxxxxxxxxx (1956–60) and xxxxed Xxxxxxxxxxx."
7,65425941,"In 1906 he was xxxxxxx xxxx to xxxxxxx xxxxxxxxx xxxx, xxxxxxing xxxxxxx a xxxxxxed xxxxx xxxxxxxx, xxxxing in xxxxxx Xxxxxxxxxx and Xxxxxx. Xxx xxxxs xxxxer, in 1908, he was xxxxxxx to xxxxxxxxxx of the Xxxxxxxxx xxxxxxxxx xxxxxxx. He was by xxx xxxxing a xxxxxxxxxx in the xxxxxx as an xxxxxxxxx xxxxxxxxx xxxxxx (or ""xxxxxxxx""). He xxxx a xxxxing xxxx in Xxxxxx xxxxxxxx xxxxxxxxxxxxxx, xxxxxxxxing xxxxxxxxxx for xxxxxxxxxx xxxxx xxxxxxxxx, which xxxxx xxxxxxxxing against the xxxxxxxx ""Xxxxxxxxxxxxxxxxxxxx"" (xxxxxxxxx ""Xxxxx-xxxxx xxxxxx xxxxxxxxxxx""), whereby the 80% of xxxxxs xxxing the xxxest xxxxxxs in xxx xxx the xxxx xxxxxs of xxxxxxxxx in xxxxxxxx xxxxxxs as the xxxxest 5% xxxing the xxxxest xxxxxxs. (The xxxxxxx xxxxxxxxxxx xxxxed xxxxxxxx over xxxx xxxxxxxxx to xxx xxxxs and xxxxxs of xxxxxxxxxx, but the xxxxxxxxing xxxxxxxxxx would xxxxxx till 1918.)"
8,40673733,"Xxxxxxxx Xxxxxxxxx Xxxxxx (Xxxx 13, 1937 – Xxxx 3, 2017) was a Xxxxxxxx-Xxxxxxxx xxxxxxxxx who xxxxxxxxxxed in the xxxxxxx of xxxxxx xxxxxxxx. She was a xxxxxxxx xxxxxxxxx at Xxxx Xxxs in Xxxxxx Xxxx, Xxx Xxxxxx from 1972 to 1997 before xxxxxxxx Xxxxxxxed Xxxxxx Xxxxxxx Xxxxxxxxxxxx in Xxxxxx, Xxx Xxxxxx. She was a xxxxxx of the Xxxxxxxx Xxxxxxxx Xxxxxxx and xxxx xxxxxxxed a Xxxxxxxxxx xxxxxxxxxx."
9,357414,"2008 xxxx xxx xxxxxxxxxx xxxxxxxxx at XXXX's Xxxxxxxx xxxxx with the xxxxxxxxxxxx of a xxx, xxxxxxx xxxxx xxxxxx. This xxxxxxxxx xxxx the Xxxxxxxx xxxxx to xxxxxxxxxxxxx 1/3 xxxxxxx xxxxxx xxxx under xxxx, not xxxxxxxing xxxxxxxx xxxxxed xxxxxxxxxx xxxed for xxxxxxxxxxx of xxxxxxxx-xxxxx xxxxxxxxxx xxxs."


#### 4.2. For POS

In [35]:
df_pos = df.copy()

# Apply POS distortion ONLY to the copy
df_pos["text"] = df_pos["text"].apply(
    lambda x: distort(
        x,
        pos=True,
        punctuation=True,
        closed_class=True,
        morphology=False,  # incompatible with pos
        capitals=True,
        reveal_numbers=True,
        enclitic=True
    )
)

df_pos = df_pos[["id", "text"]]
df_pos.to_csv("wiki_distorted_TPCRE.csv", index=False)

print("Saved POS distorted CSV")

Saved POS distorted CSV


In [36]:
df_pos.head(10)

,id,text
0,34335512,"Propn Propn Propn (Propn 11, 600 – 680s), verb adp Propn det Adj, aux det Propn noun adp det Propn noun, pron verb pron noun adp Propn adp det Propn Propn adp Adj noun."
1,43869726,Propn cconj Propn aux det 2007 Adj noun noun verb adp Propn Propn cconj verb Propn Propn cconj Propn Propn.
2,52082793,Propn Propn (verb 9 Propn 1975 adp Propn) aux det Adj noun pron verb adp det noun adp det Propn noun adj noun noun. Pron aux noun adp det noun adp det Propn Propn Propn 2001. Adp noun noun pron verb adp Propn Propn Propn adp Propn.
3,8273434,"Propn Propn Adp det Propn noun 1995 adv noun adp Propn 1994 verb sconj adv noun Propn aux part verb det noun part verb. Adp det Propn Propn pron verb 8 noun – 1999, 2009, 2013, 2015, 2017, 2019, 2021 cconj 2023. Det adv adj aux adp 2015 sconj pron verb 9th."
4,37897173,Propn noun aux det noun adp pron noun Propn. Pron aux verb adp Propn.
5,28985107,"Det noun aux adp det noun adp det noun adp Propn Propn, cconj verb adv adp det noun adp Propn, cconj adv adp det noun adp Propn."
6,29512629,"Det Propn Propn () aux num adj noun noun verb adp det Adj adj noun adv adp det Propn Propn, adp det Propn Propn adp Propn Propn Propn. Pron aux verb adp noun cconj noun noun adp det Propn Propn Propn Propn (1956–60) cconj verb Propn."
7,65425941,"Adp 1906 pron aux adv adj part verb noun noun, verb adv det adj noun noun, verb adp adj Propn cconj Propn. Num noun adv, adp 1908, pron aux verb adp noun adp det Propn adj noun. Pron aux adp adv verb det noun adp det noun adp det adj adj noun (cconj ""noun""). Pron verb det adj noun adp Propn noun noun, verb adv adp adj noun noun, pron verb verb adp det adj ""Propn"" (adv ""Num-noun noun noun""), sconj det 80% adp noun verb det adj noun adp noun verb det adj noun adp noun adp noun noun adp det adj 5% verb det adj noun. (Det adj noun verb adv adp noun verb adp noun noun cconj noun adp noun, cconj det verb noun aux verb sconj 1918.)"
8,40673733,"Propn Propn Propn (Propn 13, 1937 – Propn 3, 2017) aux det Adj-Adj noun pron verb adp det noun adp noun noun. Pron aux det noun noun adp Propn Propn adp Propn Propn, Propn Propn adp 1972 adp 1997 adp verb Propn Propn Propn Propn adp Propn, Propn Propn. Pron aux det noun adp det Propn Propn Propn cconj adv verb det Propn noun."
9,357414,"2008 adv verb adj noun adp Propn Propn noun adp det noun adp det adj, adj noun noun. Det noun verb det Propn noun adp adv 1/3 num adj noun adp noun, part verb adj verb noun verb adp noun adp adj-noun adj noun."


#### 4.3. For "blank"

In [38]:
df_blank = df.copy()

# Apply BLANK-based distortion with all options except POS and lengths (to avoid conflicts)
df_blank["text"] = df_blank["text"].apply(
    lambda x: distort(
        x,
        lengths=False,
        pos=False,
        punctuation=True,
        closed_class=True,
        morphology=True,
        capitals=True,
        reveal_numbers=True,
        enclitic=True
    )
)

df_blank = df_blank[["id", "text"]]
df_blank.to_csv("wiki_distorted_PCMKRE.csv", index=False)
print("Saved blank-based distorted CSV")

Saved blank-based distorted CSV


In [39]:
df_blank.head(10)

,id,text
0,34335512,"Blank Blank Blank (Blank 11, 600 – 680s), blank as Blank the Blank, was a Blank blank of the Blank blank, which blank its blank at Blank during the Blank Blank of Blank blank."
1,43869726,Blanks and Blanks is a 2007 Blank blank blank blank by Blank Blank and blanking Blank Blank and Blank Blank.
2,52082793,Blank Blank (blank 9 Blank 1975 in Blank) is a Blank blank who blanked as a blank for the Blank blank's blank blank blank. She was blank of the blank at the Blank Blank's Blank 2001. On blank blank she blanked for Blank Blank Blank in Blank.
3,8273434,"Blank Blank For the Blank blank 1995 blank blanks from Blank 1994 blanked so blank blank Blank did not blank a blank to blank. At the Blank Blank's they blanked 8 blanks – 1999, 2009, 2013, 2015, 2017, 2019, 2021 and 2023. The blank blank was in 2015 when they blanked 9th."
4,37897173,Blank blank is a blank of the blank Blank. It is blank on Blank.
5,28985107,"The blank is to the blank of the blank of Blank Blank, and blanks blank within the blank of Blank, and blank within the blank of Blank."
6,29512629,"The Blank Blanks () are blank blank blank blanks blanking above the Blank blank blank blank of the Blank Blanks, in the Blank Blank of Blank Blank Blank. They were blanked from blanks and blank blanks by the Blank Blank Blank Blank (1956–60) and blanked Blank."
7,65425941,"In 1906 he was blank blank to blank blank blank, blanking blank a blanked blank blank, blanking in blank Blank and Blank. Blank blanks blanker, in 1908, he was blank to blank of the Blank blank blank. He was by blank blanking a blank in the blank as an blank blank blank (or ""blank""). He blank a blanking blank in Blank blank blank, blanking blank for blank blank blank, which blank blanking against the blank ""Blank"" (blank ""Blank-blank blank blank""), whereby the 80% of blanks blanking the blankest blanks in blank blank the blank blanks of blank in blank blanks as the blankest 5% blanking the blankest blanks. (The blank blank blanked blank over blank blank to blank blanks and blanks of blank, but the blanking blank would blank till 1918.)"
8,40673733,"Blank Blank Blank (Blank 13, 1937 – Blank 3, 2017) was a Blank-Blank blank who blanked in the blank of blank blank. She was a blank blank at Blank Blanks in Blank Blank, Blank Blank from 1972 to 1997 before blank Blanked Blank Blank Blank in Blank, Blank Blank. She was a blank of the Blank Blank Blank and blank blanked a Blank blank."
9,357414,"2008 blank blank blank blank at Blank's Blank blank with the blank of a blank, blank blank blank. This blank blank the Blank blank to blank 1/3 blank blank blank under blank, not blanking blank blanked blank blanked for blank of blank-blank blank blanks."
